In [2]:
# Asyncio 51 - 100

# 51 Async Context Manager using @asynccontextmanager

import asyncio
from contextlib import asynccontextmanager


@asynccontextmanager
async def get_db_connection():
    print("Connecting to DB...")
    await asyncio.sleep(0.1)
    db = {"connected": True}
    try:
        yield db
    finally:
        print("Closing DB connection...")
        await asyncio.sleep(0.1)


async def main():
    async with get_db_connection() as db:
        print(f"DB stats: {db}")


await main()

Connecting to DB...
DB stats: {'connected': True}
Closing DB connection...


In [4]:
# 52. Chaining Async Generators

import asyncio


# Async generators allow lazy evaluation of asynchronous sequences
async def async_range(n):
    for i in range(n):
        await asyncio.sleep(0.05)  # Simulate async fetching
        yield i


# Chain async generator, memory efficient, no massive list in memory
async def async_doubled(aiter):
    async for item in aiter:
        await asyncio.sleep(0.05)  # Simulate async processing
        yield item * 2


async def main():
    # 'async for' is the idiomatic way to consume async iterables
    async for val in async_doubled(async_range(5)):
        print(val)


await main()

0
2
4
6
8


In [3]:
def add(x, y):
    return x+y
def myf(x,y,z):
    return x+y+z

In [6]:
# 53. Async Generator Cleanup (try/finally)

import asyncio


async def my_generator():
    try:
        for i in range(5):
            await asyncio.sleep(0.1)
            yield i
    finally:
        # Idiomatic place to release resources
        print("Generator cleaned up!")


async def main():
    gen = my_generator()
    async for val in gen:
        print(val)
        if val == 2:
            break  # Breaking triggers the finally block


await main()

0
1
2
Generator cleaned up!


In [9]:
# 54. Manual Future Resolution

import asyncio


async def producer(future: asyncio.Future):
    await asyncio.sleep(1)
    # A Future representing a result that will be available later
    # Use set_result() to resolve it manually
    future.set_result("The secret payload")


async def consumer(future: asyncio.Future):
    # 'await future' will pause until producer() calls set_result()
    result = await future
    print(f"Got: {result}")


async def main():
    loop = asyncio.get_running_loop()
    # Future must be bound to a running event loop
    future = loop.create_future()
    # Both tasks can now run concurrency, sharing the future as a synchronization point.
    await asyncio.gather(producer(future), consumer(future))


await main()

Got: The secret payload


In [13]:
# 55. loop.call_soon(Thread-safe Scheduling)

import asyncio


# Callbacks can be scheduled on the event loop
# This is the foundation for integrating blocking/sync code safely
def sync_callback(msg):
    print(f"Callback executed: {msg}")


async def main():
    loop = asyncio.get_running_loop()
    # call_soon() schedules the callback to run on the next iteration of the event loop
    loop.call_soon(sync_callback, "Hello immediately")

    # Must yield to the loop so it has a chance to run the scheduled callbback.
    await asyncio.sleep(0.1)
    print("Done")


await main()

Callback executed: Hello immediately
Done


In [14]:
# 56. loop.call_later (Delayed Callback)

import asyncio


def delayed_task(name):
    print(f"{name} executed")


async def main():
    loop = asyncio.get_running_loop()
    # Schedule a sync function to run after a specific delay in seconds
    loop.call_later(0.5, delayed_task, "Delayed Job")

    print("Waiting...")
    await asyncio.sleep(1)  # Keep loop alive long enough for the callback to fire
    print("Finished")


await main()

Waiting...
Delayed Job executed
Finished


In [15]:
# 57. loop.call_at(Scheduled Exact Time)

import asyncio


def scheduled_task():
    print("Exact time hit")


async def main():
    loop = asyncio.get_running_loop()
    # loop.time() returns a float representing the event loop's internal monotonic clock.
    # call+at() to schedule sometime at an exact point in time
    now = loop.time()
    loop.call_at(now + 0.5, scheduled_task)
    await asyncio.sleep(1)  # Wait for it to trigger


await main()

Exact time hit


In [17]:
# 58. Asyncio Event for Pause/Resume

import asyncio


class PausableWorker:
    def __init__(self):
        # An event is an synchronization primitive where on task can signal others
        # Setting it means "proceed", Clearing it means "block"
        self.pause_event = asyncio.Event()
        self.pause_event.set()  # Not paused initially

    async def work(self):
        for i in range(5):
            # wat() blocks if the event is cleared, resumes immediately if set
            await self.pause_event.wait()
            print(f"Working... {i}")
            await asyncio.sleep(0.2)

    def pause(self):
        self.pause_event.clear()  # Tell wait() to block

    def resume(self):
        self.pause_event.set()  # Tell wait() to unblock


async def main():
    worker = PausableWorker()
    task = asyncio.create_task(worker.work())

    await asyncio.sleep(0.3)
    worker.pause()
    print("Paused worker")
    await asyncio.sleep(1)
    worker.resume()
    print("Resumed worker")
    await task  # Wat for worker to finish


await main()

Working... 0
Working... 1
Paused worker
Resumed worker
Working... 2
Working... 3
Working... 4


In [20]:
# 59. Pub/Sub Pattern with Queues


class AsyncPubSub:
    def __init__(self):
        # Instead of a list of callable, we store subscriber Queue
        self.subscribers = asyncio.Queue()

    async def publish(self, message):
        # Fan-out: temporarily drain subscribers, send message, put them back
        subs = []
        while not self.subscribers.empty():
            subs.append(await self.subscribers.get())
        for sub in subs:
            await sub.put(message)
            await self.subscribers.put(sub)

    async def subscribe(self):
        q = asyncio.Queue()
        await self.subscribers.put(q)
        return q


async def main():
    pubsub = AsyncPubSub()
    q1 = await pubsub.subscribe()
    q2 = await pubsub.subscribe()

    asyncio.create_task(pubsub.publish("News"))

    # Subscribers simply await their own queues, fully decoupled
    print("Sub 1:", await q1.get())
    print("Sub 2:", await q2.get())


await main()

Sub 1: News
Sub 2: News


In [21]:
# 60. run_coroutine_threadsafe (Corss-thread)

import asyncio
import threading
import time


async def async_coroutine(val):
    await asyncio.sleep(0.1)
    print(f"Coroutine executed with: {val}")


def thread_worker(loop):
    time.sleep(0.1)
    # Idiomatic solution: r#un_coroutine_threadsafe safely injects the coroutine
    # back into the event loop's thread and returns a concurrent.futures.Future.
    future = asyncio.run_coroutine_threadsafe(async_coroutine(42), loop)
    future.result()  # Block the background thread until the async function finishes


async def main():
    loop = asyncio.get_running_loop()
    thread = threading.Thread(target=thread_worker, args=(loop,))
    thread.start()
    await asyncio.sleep(0.5)  # Keep loop alive
    thread.join()


await main()

Coroutine executed with: 42


In [23]:
# . 61 StreamReader (realine)

import asyncio


async def tcp_client():
    reader, writer = await asyncio.open_connection("example.com", 80)
    writer.write(b"GET / HTTP/1.1\r\nHost: example.com\r\n\r\n")
    await writer.drain()

    # StreamReader provides high-level methods over raw sockets.
    # readline() is idiomatic for parsing line-based protocols (like HTTP headers).
    # It reads until it hits a b'\n' and returns the line.
    while True:
        line = await reader.readline()
        if line == b"\r\n":  # Empty line denotes end of HTTP headers
            break
        print(line.decode().strip())


await tcp_client()

HTTP/1.1 200 OK
Date: Sat, 11 Apr 2026 16:24:31 GMT
Content-Type: text/html
Transfer-Encoding: chunked
Connection: keep-alive
Server: cloudflare
Last-Modified: Fri, 10 Apr 2026 01:29:17 GMT
Allow: GET, HEAD
Accept-Ranges: bytes
Age: 3136
cf-cache-status: HIT
CF-RAY: 9eab578e0b9f7ac1-SJC


In [24]:
# 62. StreamReader (readuntil)

import asyncio


async def read_until_separator(reader: asyncio.StreamReader, sep: bytes):
    # readuntil() is highly idiomatic for binary protocols (e.g., MQTT, custom TCP)
    # where messages are delimited by a specific byte sequence, not just newlines.
    while True:
        chunk = await reader.readuntil(sep)
        if not chunk:
            break
        print("chunk:", chunk.strip())


# Note: readuntil raises IncompleteReadError if EOF is reached before the separator

In [25]:
# 63. StreamReader (readexactly)

import asyncio


async def read_exact_header():
    # We can mock a StreamReader by feeding it data manually
    reader = asyncio.StreamReader()
    reader.feed_data(b"1234567890EXTRA")
    reader.feed_eof()

    # readexactly() is used when you know the exact byte count of the payload
    # (e.g., reading a 4-byte integer length prefix). It blocks until exactly N bytes are read.
    # If EOF is hit first, it raises IncompleteReadError.
    first_ten = await reader.readexactly(10)
    print(f"Exact 10 bytes: {first_ten}")

    rest = await reader.read()
    print(f"Rest: {rest}")


await read_exact_header()

Exact 10 bytes: b'1234567890'
Rest: b'EXTRA'


In [ ]:
# 64. Graceful Server Shutdown via Signal

import asyncio
import signal


async def handle_client(reader, writer):
    await asyncio.sleep(1)
    writer.close()
    await writer.wait_closed()


async def main():
    server = await asyncio.start_server(handle_client, "127.0.0.1", 8889)

    loop = asyncio.get_running_loop()
    stop = loop.create_future()

    # Idiomatic graceful shutdown: instead of relying on try/except KeyboardInterrupt
    # at the top level (which doesn't work cleanly inside asyncio), we bind OS signals
    # (SIGINT is Ctrl+C) to resolve a Future.
    loop.add_signal_handler(signal.SIGINT, stop.set_result, None)
    loop.add_signal_handler(signal.SIGTERM, stop.set_result, None)

    print("Server running. Press Ctrl+C to stop.")
    async with server:
        await stop  # The loop sits here until a signal sets the Future's result

    print("Shutting down gracefully.")


await main()

Server running. Press Ctrl+C to stop.


In [1]:
# 65. Async File I/O via Executor

import asyncio
import os


async def async_read_file(path):
    loop = asyncio.get_running_loop()

    def read():
        with open(path, "r") as f:
            return f.read()

    # Python's built-in file I/O is blocking. While small reads are fast, large file
    # reads can block the event loop. The idiomatic workaround (prior to aiofiles) is
    # run_in_executor(None, ...) which pushes the blocking function to a ThreadPool.
    content = await loop.run_in_executor(None, read)
    print(f"Read {len(content)} chars")


async def main():
    with open("test.txt", "w") as f:
        f.write("A" * 100)
    await async_read_file("test.txt")
    os.remove("test.txt")


await main()

Read 100 chars


In [ ]:
# 66. ProcessPoolExecutor with asyncio

import asyncio
from concurrent.futures import ProcessPoolExecutor


def cpu_heavy(n):
    return sum(i * i for i in range(n))


async def main():
    loop = asyncio.get_running_loop()
    # run_in_executor with a ThreadPoolExecutor still respects the GIL for CPU-bound Python code.
    # Idiomatic way to bypass the GIL entirely for CPU-heavy work is using a ProcessPoolExecutor.
    # This actually spawns separate Python processes.
    with ProcessPoolExecutor() as pool:
        results = await asyncio.gather(
            loop.run_in_executor(pool, cpu_heavy, 10 ** 6),
            loop.run_in_executor(pool, cpu_heavy, 10 ** 6),
        )
    print(results)


# await main()
# Run in terminal to test
# if __name__ == '__main__':
#     asyncio.run(main())

In [12]:
# 67. Async LRU Cache Implementation

import asyncio
from functools import wraps


# Standard functools.lru_cache does NOT work with async functions because it attempts
# to hash the coroutine object, not the result. We must build an async-aware LRU.
def async_lru(maxsize=3):
    cache = {}
    queue = asyncio.Queue(maxsize=maxsize)

    def decorator(func):
        @wraps(func)
        async def wrapper(*args):
            if args in cache:
                return cache[args]
            if queue.full():
                oldest = await queue.get()  # Evict oldst
                cache.pop(oldest, None)
            result = await func(*args)
            cache[args] = result
            await queue.put(args)
            return result

        return wrapper

    return decorator


@async_lru(maxsize=2)
async def fetch(id):
    await asyncio.sleep(0.1)
    return f"Data-{id}"


async def main():
    print(await fetch(1))  # Miss
    print(await fetch(2))  # Miss
    print(await fetch(1))  # Hit


await main()

Data-1
Data-2
Data-1


In [13]:
# 68. Token Bucket Rate Limiter


class TokenBucket:
    def __init__(self, rate: float, capacity: int = 1):
        self.rate = rate  # Tokens added per second
        self.tokens = capacity
        self.max_tokens = capacity
        self.last_refill = asyncio.get_event_loop().time()

    async def acquire(self):
        # Token bucket is the standard algorithm for rate limiting APIs.
        # It allows bursts up to 'capacity', but enforces a sustained 'rate'.
        while self.tokens < 1:
            self._refill()
            await asyncio.sleep(0.1)  # Yield control while waiting for tokens
        self.tokens -= 1
        self._refill()

    def _refill(self):
        now = asyncio.get_event_loop().time()
        elapsed = now - self.last_refill
        self.tokens = min(self.max_tokens, self.tokens + elapsed * self.rate)
        self.last_refill = now


async def main():
    bucket = TokenBucket(rate=2.0)
    for i in range(5):
        await bucket.acquire()
        print(f"Request {i} at {asyncio.get_event_loop().time():.2f}")


await main()

Request 0 at 1064174.68
Request 1 at 1064175.28
Request 2 at 1064175.79
Request 3 at 1064176.30
Request 4 at 1064176.80


In [4]:
# 69. Fan-out, Fan-in Pattern

import asyncio


async def fetch_sub(url, sub):
    await asyncio.sleep(0.1)
    return f"{url}/sub{sub}"


async def fan_out(url):
    await asyncio.sleep(0.1)
    # Fan-out: One task spawns multiple sub-tasks.
    # Note: We return the tasks themselves, NOT awaiting them here.
    return [asyncio.create_task(fetch_sub(url, i)) for i in range(3)]


async def main():
    urls = ["a.com", "b.com"]

    # First layer of gather fans out
    subtask_lists = await asyncio.gather(*[fan_out(u) for u in urls])

    # Flatten the list of lists
    all_subtasks = [t for sublist in subtask_lists for t in sublist]

    # Fan-in: gather all sub-tasks concurrently and wait for them all
    results = await asyncio.gather(*all_subtasks)
    print(results)


# asyncio.run(main())
await main()

['a.com/sub0', 'a.com/sub1', 'a.com/sub2', 'b.com/sub0', 'b.com/sub1', 'b.com/sub2']


In [12]:
# 70. Async Map (Gather)

import asyncio


# Idiomatic equivalent to Python's built-in map(), but runs concurrently.
async def async_map(coro_func, iterable):
    # Unpack a generator expression into gather. This schedule all tasks immediately.
    return await asyncio.gather(*(coro_func(item) for item in iterable))


async def square(n):
    await asyncio.sleep(0.1)
    return n ** 2


async def main():
    # This task ~0.1s total, not 0.5s, because all run concurrently
    results = await async_map(square, range(5))
    print(results)


await main()

[0, 1, 4, 9, 16]


In [16]:
# 71. Async Map (as_completed)

import asyncio

# Variation of async_map that yields results as soon as they finish
# regardless of the original input order


async def async_map_unordered(coro_func, iterable):
    tasks = [asyncio.create_task(coro_func(item)) for item in iterable]
    # as_completed yields futures in the order they finish
    for task in asyncio.as_completed(tasks):
        yield await task


async def fetch(n):
    await asyncio.sleep(1.0 - (n * 0.1))  # High numbers finish faster
    return n


async def main():
    async for result in async_map_unordered(fetch, range(5)):
        print(result)


await main()

4
3
2
1
0


In [18]:
# 72. Batching Async Iterables

import asyncio


# When streaming thousands of items from a DB, you often want to process them in
# chunks (e.g., to send bulk inserts to another database)
async def batched(async_iter, batch_size):
    batch = []
    async for item in async_iter:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch  # Yield full batch
            batch = []
    if batch:
        yield batch  # Yield remaining partial batch


async def item_generator(n):
    for i in range(n):
        await asyncio.sleep(0.5)
        yield i


async def main():
    async for batch in batched(item_generator(7), 3):
        print("Batch:", batch)


await main()

Batch: [0, 1, 2]
Batch: [3, 4, 5]
Batch: [6]


In [20]:
# 73. Queue.get with Timeout

import asyncio


async def consumer(queue):
    try:
        # queue.get() blocks forever by default. Wrapping it in wait_for is the
        # idiomatic way to implement "timeout if no message arrives within X seconds".
        item = await asyncio.wait_for(queue.get(), timeout=0.5)
        print(f"Got: {item}")
    except asyncio.TimeoutError:
        print("Queue was empty, timed out!")


async def main():
    queue = asyncio.Queue()
    await consumer(queue)  # No producer, so it times out


await main()

Queue was empty, timed out!


In [23]:
# 74. Lock.acquire with Timeout

import asyncio

lock = asyncio.Lock()


async def try_lock(name):
    try:
        # While 'async with lock:' is standard, sometimes you want to give up
        # if the lock isn't available immediately. We can do this by wrapping
        # the underlying acquire() coroutine in wait_for.
        await asyncio.wait_for(lock.acquire(), timeout=0.5)
        print(f"{name} acquired lock")
        await asyncio.sleep(2)
    except asyncio.TimeoutError:
        print(f"{name} failed to acquire lock")
    finally:
        if lock.locked():
            lock.release()


async def main():
    await asyncio.gather(try_lock("Task-1"), try_lock("Task-2"))


await main()

Task-1 acquired lock
Task-2 failed to acquire lock


In [25]:
# 75. Watchdog / Heartbeat Task

import asyncio


async def worker():
    for i in range(3):
        await asyncio.sleep(0.5)
        print("Worker heartbeat")


async def watchdog(timeout):
    # A watchdog simply waits for a specific time. If it wakes up naturally,
    # the worker failed to cancel it, meaning the worker is stuck/deadlocked.
    try:
        await asyncio.sleep(timeout)
        print("Watchdog timed out! Worker is dead.")
    except asyncio.CancelledError:
        print("Watchdog cancelled (Worker finished).")


async def main():
    task = asyncio.create_task(worker())
    wd = asyncio.create_task(watchdog(2.0))

    # wait with FIRST_COMPLETED is perfect for "wait for A, but cancel B if A finishes first".
    done, pending = await asyncio.wait([task, wd], return_when=asyncio.FIRST_COMPLETED)

    for p in pending:
        p.cancel()
        await p  # Suppress CancelledError


await main()

Worker heartbeat
Worker heartbeat
Worker heartbeat
Watchdog cancelled (Worker finished).


In [27]:
# 76. Interruptible asyncio.sleep

import asyncio


async def interruptible_sleep(task_to_cancel):
    try:
        await asyncio.sleep(20)
    except asyncio.CancelledError:
        # Intercepting CancelledError is the idiomatic way to create "cancellable delays".
        # Here we use the cancellation of one task to trigger the cancellation of another.
        print("Sleep interrupted")
        task_to_cancel.cancel()
        raise  # Always re-raise CancelledError unless you are strictly suppressing it


async def main_work():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("Main work cancelled.")


async def main():
    main_task = asyncio.create_task(main_work())
    sleep_task = asyncio.create_task(interruptible_sleep(main_task))

    await asyncio.sleep(0.5)
    sleep_task.cancel()  # Trigger the chain reaction

    try:
        await sleep_task
    except asyncio.CancelledError:
        pass


await main()

Sleep interrupted
Main work cancelled.


In [34]:
# 77. Cooperatively Yielding cpu (Chunked Processing)

import asyncio


async def process_large_list(items):
    for i in range(0, len(items), 100):
        chunk = items[i : i + 100]
        # Simulate heavy CPU work (e.g., parsing JSON, image processing)
        for _ in chunk:
            pass

        # Problem: CPU-bound work blocks the event loop, preventing other tasks from running.
        # Idiomatic solution without Executors: process in small chunks and yield control
        # to the loop using await asyncio.sleep(0). This prevents UI/network freezing.
        await asyncio.sleep(0)


async def main():
    await process_large_list(list(range(10000)))
    print("Processed without blocking event loop")


await main()

Processed without blocking event loop


In [36]:
# 78. Subprocess Streaming with Timeout

import asyncio


async def run_with_timeout(cmd, timeout):
    proc = await asyncio.create_subprocess_shell(
        cmd, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE
    )

    try:
        stdout, stderr = await asyncio.wait_for(proc.communicate(), timeout=timeout)
        print(stdout.decode())
    except asyncio.TimeoutError:
        # CRITICAL: When timing out a subprocess, you MUST manually kill it.
        # Otherwise, it becomes a zombie process running in the background.
        proc.kill()
        await proc.communicate()  # Reap the process to prevent zombies
        print("Process killed due to timeout")


async def main():
    await run_with_timeout("sleep 5 && echo done", timeout=1.0)


await main()

Process killed due to timeout


In [39]:
# 79. Merging Multiple Async Iterators

import asyncio


# Merging async iterators is tricky because they yield at unpredictable times.
# The most robust Pythonic way is using an intermediate Queue to collect items
# as they arrive, regardless of which iterator produced them.
async def merge_aiters(*aiters):
    queue = asyncio.Queue()
    sentinel = object()  # Unique marker to know when an iterator finishes

    async def producer(aiter):
        async for item in aiter:
            await queue.put(item)
        await queue.put(sentinel)

    tasks = [asyncio.create_task(producer(a)) for a in aiters]

    active = len(tasks)
    while active > 0:
        item = await queue.get()
        if item is sentinel:
            active -= 1
        else:
            yield item


async def gen1():
    for i in [1, 4, 5]:
        yield i


async def gen2():
    for i in [2, 3, 4]:
        yield i


async def main():
    async for val in merge_aiters(gen1(), gen2()):
        print(val)


await main()

1
4
5
2
3
4


In [40]:
# 80. Zipping Async Iterators
import asyncio


async def azip(*aiters):
    # Standard zip() waits for both. To do this asynchronously without blocking,
    # We gather the next item from ALL iterators concurrently
    iters = [ait.__aiter__() for ait in aiters]
    while True:
        # Run all __anext__ calls in parallel. If ANY of them raises StopAsyncIteration,
        # return_exceptions=True catches it so we can cleanly break the loop.
        results = await asyncio.gather(
            *(it.__anext__() for it in iters), return_exceptions=True
        )
        if any(isinstance(r, StopAsyncIteration) for r in results):
            break
        yield tuple(results)


async def main():
    async def nums():
        for i in [1, 2, 3]:
            yield i

    async def lets():
        for i in ["a", "b", "c"]:
            yield i

    async for pair in azip(nums(), lets()):
        print(pair)


await main()

(1, 'a')
(2, 'b')
(3, 'c')


In [42]:
# 81. Async Abstract Base Class (ABC)
import asyncio
from abc import ABC, abstractmethod


# Using ABCs with async methods enforces a contract for async implementations.
# This is idiomatic for defining interfaces for async repositories, clients, etc
class AsyncDatabase(ABC):
    @abstractmethod
    async def connect(self):
        pass

    @abstractmethod
    async def query(self, sql):
        pass


class FakeDB(AsyncDatabase):
    async def connect(self):
        await asyncio.sleep(0.1)
        print("Connected")

    async def query(self, sql):
        await asyncio.sleep(0.1)
        return f"Result of {sql}"


async def main():
    db: AsyncDatabase = FakeDB()
    await db.connect()
    print(await db.query("SELECT 1"))


await main()

Connected
Result of SELECT 1


In [43]:
# 82. Dependency Injection in Async Setup

import asyncio


# Dependency Injection (DI) works exactly the same in async as it does in sync.
# It keeps your code testable and decoupled. Pass the dependency BEFORE starting async work.
class Service:
    def __init__(self, repo):
        self.repo = repo  # Injected dependency

    async def get_data(self):
        return await self.repo.fetch()


class Repository:
    async def fetch(self):
        await asyncio.sleep(0.1)
        return "Data"


async def main():
    repo = Repository()
    service = Service(repo)
    print(await service.get_data())


# asyncio.run(main())
await main()

Data


In [47]:
# 83. Async Singleton Client
import asyncio


# Creating an async singleton is harder than a sync one because __init__ cannot be async.
# Idiomatic solution: raise an error in __init__, provide an async initialize() classmethod,
# and use an asyncio.Lock to prevent race conditions during initialization.
class AsyncSingleton:
    _instance = None
    _lock = asyncio.Lock()

    def __new__(cls):
        if cls._instance is None:
            raise RuntimeError("Use AsyncSingleton.initialize()")
        return cls._instance

    @classmethod
    async def initialize(cls):
        async with cls._lock:  #  Ensure only one task initializes it
            if cls._instance is None:
                cls._instance = super().__new__(cls)
                await asyncio.sleep(0.1)  # Simulate heavy init
                cls._instance.state = "ready"
        return cls._instance


async def main():
    s1 = await AsyncSingleton.initialize()
    s2 = await AsyncSingleton.initialize()
    print(s1 is s2)  # Tru
    print(s1.state, s2.state, id(s1), id(s2))


await main()

True
ready ready 4381045632 4381045632


In [48]:
# 84. ExceptionGroup Handling (Python 3.11+ except*)
import asyncio


async def raise_value():
    raise ValueError("Val error")

async def raise_type():
    raise TypeError("Type erorr")

async def main():
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(raise_value())
            tg.create_task(raise_type())
        # Python 3.11 introduced ExceptionGroup (when multiple tasks fail in TaskGroup).
        # The idiomatic way to handle them is using 'except*', which filters by exception type.
    except* ValueError as eg:
        print(f"Handled Values: {eg.exceptions}")
    except* TypeError as eg:
        print(f"Handled Types: {eg.exceptions}")

await main()

Handled Values: (ValueError('Val error'),)
Handled Types: (TypeError('Type erorr'),)


In [50]:
# 85. asyncio.wait with Specific Timeout

import asyncio


async def worker(n):
    await asyncio.sleep(n)
    return n


async def main():
    tasks = [asyncio.create_task(worker(i)) for i in range(1, 6)]

    # asyncio.wait() allows setting a timeout that applies to the ENTIRE group.
    # Unlike wait_for, it doesn't raise an exception; it just returns whatever is done
    done, pending = await asyncio.wait(tasks, timeout=2.5)

    print(f"Completed {len(done)} tasks")
    for t in done:
        print(f" - {t.result()}")

    # You are responsible for cleaning up the pending tasks:
    print(f"Cancelled {len(pending)} tasks")
    for t in pending:
        t.cancel()


await main()

Completed 2 tasks
 - 2
 - 1
Cancelled 3 tasks


In [51]:
# 86. Safely Cancelling Pending Tasks Helper

import asyncio

# When cancelling multiple tasks, calling task.cancel() only *requests* cancellation.
# The task must still be awaited to allow it to raise CancelledError and run its finally blocks.
# Idiomatic pattern: gather with return_exceptions=True to silently absorb the CancelledErrors.


async def cancel_pending(tasks):
    for task in tasks:
        task.cancel()
    await asyncio.gather(*tasks, return_exceptions=True)


async def main():
    tasks = [asyncio.create_task(asyncio.sleep(100)) for _ in range(5)]
    await asyncio.sleep(0.5)
    await cancel_pending(tasks)
    print("All pending tasks safely cancelled")


await main()

All pending tasks safely cancelled


In [53]:
# 87. Re-creating a Task on Failure (Retry Loop)

import asyncio
import random


async def flaky_api():
    if random.random() < 0.8:
        raise ConnectionError("Lost connection")
    return "Success"


# Instead of complex retry decorators, a simple while-true loop with try/except
# is often more Pythonic and easier to debug for long-running retryable tasks.
async def persistent_task():
    while True:
        try:
            return await flaky_api()
        except ConnectionError:
            print("Restarting task...")
            await asyncio.sleep(0.1)


async def main():
    result = await persistent_task()
    print(result)


await main()

Restarting task...
Restarting task...
Success


In [54]:
# 88. Async State Machine

import asyncio


# State machines are highly compatible with async/await because await naturally
# pauses the machine until an event (like an Event object or Queue) triggers a transition.
class AsyncStateMachine:
    def __init__(self):
        self.state = "idle"
        self.event = asyncio.Event()

    async def run(self):
        while True:
            if self.state == "idle":
                print("State: IDLE, Waiting...")
                await self.event.wait()  # Pause machine
                self.event.clear()
                self.state = "processing"
            elif self.state == "processing":
                print("State: PROCESSING")
                await asyncio.sleep(1)
                self.state = "idle"

    def trigger(self):
        self.event.set()  # External trigger wakes the machine


async def main():
    sm = AsyncStateMachine()
    task = asyncio.create_task(sm.run())

    sm.trigger()
    await asyncio.sleep(0.5)
    sm.trigger()
    await asyncio.sleep(1.5)

    task.cancel()


await main()

State: IDLE, Waiting...
State: PROCESSING
State: IDLE, Waiting...
State: PROCESSING


In [59]:
# 89. Avoiding create_task in init

import asyncio


# ANTI-PATTERN: Starting a task inside __init__.
# __init__ is synchronous. If the event loop isn't running yet, this crashes.
# Even if it is running, the taks starts BEFORE __init__ finishes setting up variables.
class BadService:
    def __init__(self):
        self.task = asyncio.create_task(self.run())
        self.data = "initialized"  # Too late! run(0 already started without this!


# IDIOMATIC PATTERN: Explicit lifecycle methods
class GoodService:
    def __init__(self):
        self.data = "initialized"
        self.task = None

    async def start(self):
        # Safe: called after object is fully constructed and loop is running
        self.task = asyncio.create_task(self.run())

    async def run(self):
        print(f"Running with: {self.data}")


async def main():
    service = GoodService()
    await service.start()
    await service.task


await main()

Running with: initialized


In [60]:
# 90. Async Context Manager with contextlib.suppress

import asyncio
from contextlib import asynccontextmanager


# You can use standard contextlib tools inside @asynccontextmanager.
# This is clean for operations where you want to swallow exceptions entirely.
@asynccontextmanager
async def risky_operation():
    print("Attempting risky op")
    try:
        yield
    except Exception:
        # The exception is caught here. The caller's 'async with' block will NOT
        # see the exception. Use with caution.
        print("Cleaned up internally!")


async def main():
    async with risky_operation():
        raise ValueError("Oops")  # This is suppressed


await main()

Attempting risky op
Cleaned up internally!


In [61]:
# 91. Getting Loop Time
import asyncio


async def measure_time():
    loop = asyncio.get_running_loop()
    start = loop.time()
    await asyncio.sleep(1.0)
    end = loop.time()

    # ALWAYS use loop.time() for internal asyncio measurements.
    # time.time() uses the system wall clock, which can jump backwards/forwards
    # if the OS updates the clock (NTP). loop.time() is strictly monotonic.
    print(f"Elapsed: {end - start:.2f}s")


await measure_time()

Elapsed: 1.00s


In [73]:
# 92. Weakref with Tasks
import asyncio
import gc  # Import the garbage collector interface
import weakref


async def my_task():
    await asyncio.sleep(1)


async def main():
    task = asyncio.create_task(my_task())
    # weakref.ref creates a reference that does NOT prevent the garbage collector
    # from destroying the object. Idiomatic for caches or tracking tasks without
    # preventing memory leaks if you forget to remove them from a list.
    weak_task = weakref.ref(task)

    await task
    del task  # Remove the strong reference
    # Manually trigger garbage collection
    # gc.collect()

    # # 2. Clear IPython's internal reference to the last result
    # import sys
    # if hasattr(sys, 'last_value'): del sys.last_value
    # if hasattr(sys, 'last_traceback'): del sys.last_traceback

    # # 3. Force collection multiple times (sometimes needed for cycle detection)
    # gc.collect()
    # gc.collect()

    # The weak reference is now dead because the task finished and was garbage collected
    print(f"Weakref exists? {weak_task() is not None}")


await main()
# True due to hidden referenes in notebook

Weakref exists? True


In [75]:
# 93. Periodic Task Execution

import asyncio


# To run a task every X seconds, you must be careful that the task execution time
# doesn't "drift" the schedule. Idiomatic approach: gather the task and the sleep together.
async def periodic(internal, func, *args):
    while True:
        # If func takes 0.5s and interval is 1.0s, this gather ensures the next
        # iteration starts exactly 1.0s after the PREVIOUS iteration STARTED.
        await asyncio.gather(func(*args), asyncio.sleep(internal))


async def tick():
    print(f"Tick at {asyncio.get_event_loop().time():.2f}")


async def main():
    task = asyncio.create_task(periodic(1.0, tick))
    await asyncio.sleep(3.5)  # Let it run a few times
    task.cancel()  # Stop it


await main()

Tick at 1226226.66
Tick at 1226227.66
Tick at 1226228.66
Tick at 1226229.67


In [77]:
# 94. Debouncing Async Calls
import asyncio


# Debouncing: If a function is called rapidly, only execute it ONCE after the
# calls stop for a specific delay. Crucial for autocomplete/search bars.
class AsyncDebouncer:
    def __init__(self, delay):
        self.delay = delay
        self.task = None

    async def call(self, func, *args):
        # If a task is already pending, cancel it
        if self.task and not self.task.done():
            self.task.cancel()
        # Start a new delayed task
        self.task = asyncio.create_task(self._wrapper(func, *args))

    async def _wrapper(self, func, *args):
        await asyncio.sleep(self.delay)
        await func(*args)  # Only this final call will actually execute


async def print_val(val):
    print(f"Executed with: {val}")


async def main():
    debouncer = AsyncDebouncer(0.5)

    # Spam calls rapidly,
    for i in range(10):
        await debouncer.call(print_val, i)
        await asyncio.sleep(0.2)

    await asyncio.sleep(1)  # Wait for the final debounce to fire


await main()

Executed with: 9


In [79]:
# 95. Throttling Async Calls
import asyncio


# Throttling: Ensure a function never runs more than N times concurrently.
# This is simply a wrapper around asyncio.Semaphore.
class AsyncThrottler:
    def __init__(self, rate_limit):
        self.sem = asyncio.Semaphore(rate_limit)

    async def call(self, func, *args):
        async with self.sem:  # Blocks if limit is reached
            return await func(*args)


async def api_call(n):
    print(f"Start {n}")
    await asyncio.sleep(1)
    print(f"End {n}")
    return n


async def main():
    throttler = AsyncThrottler(2)  # Max 2 at a time
    # 5 tasks spawned, but only 2 will run. As one finishes, the next starts.
    await asyncio.gather(*[throttler.call(api_call, i) for i in range(5)])


await main()

Start 0
Start 1
End 0
End 1
Start 2
Start 3
End 2
End 3
Start 4
End 4


In [80]:
# 96. Wrapping Sync Iterables to Async
import asyncio

# Sometimes you are given a standard synchronous iterator but need to use it
# inside an async context without blocking the loop.


async def to_async_iterator(sync_iter):
    for item in sync_iter:
        # The magic is 'await asyncio.sleep(0)'. It yields control to the event loop
        # at every step, allowing other tasks to run, turning a sync generator async.
        await asyncio.sleep(0)
        yield item


async def main():
    sync_list = range(3)
    async for val in to_async_iterator(sync_list):
        print(val)


await main()

0
1
2


In [81]:
# 97. Collectiing Results into a Dict via gather
import asyncio


async def fetch_user(uid):
    await asyncio.sleep(0.1 * uid)
    return {"id": uid, "name": f"User{uid}"}


async def main():
    uids = [1, 2, 3]
    # Problem: asyncio.gather returns a list. If you need to map results back to inputs.
    # you lose the association if the order isn't strictly 1:1 or if you use as_completed.
    # Idiomatic solution: keep a dict of Task objects mapped to their inpup ID.
    tasks = {uid: asyncio.create_task(fetch_user(uid)) for uid in uids}

    # Await them and map back using dict comprehension
    results = {uid: await task for uid, task in tasks.items()}
    print(results)


await main()

{1: {'id': 1, 'name': 'User1'}, 2: {'id': 2, 'name': 'User2'}, 3: {'id': 3, 'name': 'User3'}}


In [84]:
# 98. Basic Asyncio Actor Model
import asyncio


# The Actor Model: concurrency via isolated "actors" that communicate ONLY by
# sending messages to each other's mailboxes (Queues). No shared memory = no locks needed.
class Actor:
    def __init__(self, name):
        self.name = name
        self.mailbox = asyncio.Queue()  # The mailbox
        self.task = None

    async def start(self):
        self.task = asyncio.create_task(self._process())

    async def _process(self):
        while True:
            msg = await self.mailbox.get()  # Wait for work
            if msg == "STOP":
                break
            print(f"{self.name} received: {msg}")

    async def send(self, msg):
        # Sending a msg is non-blocking: it just puts it in the queue
        await self.mailbox.put(msg)


async def main():
    alice = Actor("Alice")
    bob = Actor("Bob")

    await asyncio.gather(alice.start(), bob.start())

    await alice.send("Hello Bob")
    await bob.send("Hello Alice")
    await alice.send("STOP")
    await bob.send("STOP")

    await asyncio.gather(alice.task, bob.task)


await main()

Alice received: Hello Bob
Bob received: Hello Alice


In [89]:
# Dynamic Callback Addition

# An Event Emitter pattern allows dynamically attaching functions to events.
# We use a queue of callbacks so we can modify the listener list while iterating.
class AsyncEventEmitter:
    def __init__(self):
        self.listeners = asyncio.Queue()

    def on(self, callback):
        # Registering a listener just adds it to the queue
        self.listeners.put_nowait(callback)

    async def emit(self, *args):
        callbacks = []
        # Drain the queue
        while not self.listeners.empty():
            callbacks.append(self.listeners.get_nowait())
        # Fire them concurrently (don't await them here, or emit blocks!)
        for cb in callbacks:
            asyncio.create_task(cb(*args))


async def on_data(data):
    await asyncio.sleep(0.1)
    print(f"Processed: {data}")


async def main():
    emitter = AsyncEventEmitter()
    emitter.on(on_data)
    emitter.on(on_data)  # Register twice = runs twice

    await emitter.emit("my_payload")
    await asyncio.sleep(0.5)  # Yield so background tasks can finish


await main()

Processed: my_payload
Processed: my_payload


In [6]:
# 99. Dynamic Callback Addition

import asyncio


# An Event Emitter pattern allows dynamically attaching functions to events.
# We use a queue of callbacks so we can modify the listener list while iterating.
class AsyncEventEmitter:
    def __init__(self):
        self.listeners = asyncio.Queue()

    def on(self, callback):
        # Registering a listener just adds it to the queue
        self.listeners.put_nowait(callback)

    async def emit(self, *args):
        callbacks = []
        # Drain the queue
        while not self.listeners.empty():
            callbacks.append(self.listeners.get_nowait())
        # Fire them concurrently (don't await them here, or emit blocks!)
        for cb in callbacks:
            asyncio.create_task(cb(*args))


async def on_data(data):
    await asyncio.sleep(0.1)
    print(f"Processed: {data}")


async def main():
    emitter = AsyncEventEmitter()
    emitter.on(on_data)
    emitter.on(on_data)  # Register twice = runs twice

    await emitter.emit("my_payload")
    await asyncio.sleep(0.5)  # Yield so background tasks can finish


await main()

Processed: my_payload
Processed: my_payload


In [7]:
# 100. Task Naming and Filtering
import asyncio


async def worker(worker_id):
    await asyncio.sleep(1)
    return worker_id


async def monitor():
    while True:
        # asyncio.all_tasks() returns ALL tasks, including the current one and main.
        # Idiomatic way to inspect them: name your tasks and filter by prefix.
        tasks = asyncio.all_tasks()
        worker_tasks = [t for t in tasks if t.get_name().startswith("worker-")]
        print(f"Active workers: {len(worker_tasks)}")
        await asyncio.sleep(0.5)
        if not worker_tasks:
            break


async def main():
    monitor_task = asyncio.create_task(monitor(), name="monitor")

    # The 'name' kwarg is crucial for debugging and task management in complex apps.
    for i in range(5):
        asyncio.create_task(worker(i), name=f"worker-{i}")

    await monitor_task  # Wait until monitor detects 0 workers


await main()

Active workers: 5
Active workers: 5
Active workers: 0
